# Эксперимент 30: Whisper adaptation

**Статья:** Adaptation of Whisper models to child speech recognition (Адаптация моделей Whisper к распознаванию детской речи) 2023

**Ссылка:** [https://arxiv.org/abs/2307.13008](https://arxiv.org/abs/2307.13008)

**Краткое описание модели:** Whisper encoder embeddings (openai/whisper-small) с временной агрегацией статистик и discriminative classifier.

**Содержание статьи:** Используются реальные Whisper-представления вместо handcrafted proxy, что приближает эксперимент к статье об адаптации Whisper к детской речи.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report
from transformers import WhisperProcessor, WhisperModel

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

I0000 00:00:1775023067.197660   55595 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Загрузка данных и разбиение

In [2]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [3]:
MODEL_ID = "openai/whisper-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = WhisperProcessor.from_pretrained(MODEL_ID)
whisper = WhisperModel.from_pretrained(MODEL_ID).to(device)
whisper.eval()

def extract_embed(path):
    y, sr = data_utils.load_audio(path, sr=16000, max_sec=config.MAX_DURATION_SEC)
    feats = processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        hs = whisper.encoder(input_features=feats.input_features.to(device)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(axis=0), hs.std(axis=0), hs.max(axis=0)], axis=0).astype(np.float32)

X_train = np.stack([extract_embed(p) for p in paths_train])
X_val   = np.stack([extract_embed(p) for p in paths_val])
X_test  = np.stack([extract_embed(p) for p in paths_test])

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])


## 3. Обучение, оценка и сохранение результата

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=config.RANDOM_STATE))
])

param_grid = {
    "clf__C": [0.01, 0.1, 0.5, 0.7, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1, 0.5],
}

t0 = time.perf_counter()

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

train_time_sec = time.perf_counter() - t0

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_30_whisper_svm", 
    experiment_name="Whisper encoder SVM", 
    model="Whisper embedding + SVM", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"whisper_small_encoder/grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=train_time_sec
)

Fitting 5 folds for each of 63 candidates, totalling 315 fits
              precision    recall  f1-score   support

        good       0.86      0.90      0.88       282
         bad       0.77      0.68      0.72       135

    accuracy                           0.83       417
   macro avg       0.81      0.79      0.80       417
weighted avg       0.83      0.83      0.83       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.829736,0.799472,0.721569,0.890334,0.766667,0.681481


PosixPath('/mnt/d/Projects/HSE/VKR/VKR/experiments/exp_30_whisper_svm/result.csv')